In [ ]:
# =============================================================================
# BIBM E3 评测模板 (eval_template_cnn.ipynb)
# =============================================================================
# 用法 (在服务器上):
#   1. cp eval_template_cnn.ipynb eval_bibm_e3_<backbone>_<setting>.ipynb
#      例: cp eval_template_cnn.ipynb eval_bibm_e3_unet_ubl.ipynb
#   2. 改下面的 CKPT_PATH 和 BACKBONE 两行
#   3. jupyter 跑 → final_summary 表
# 注意:
#   - 复制出的 eval_*.ipynb 不进 git (已 .gitignore)
#   - 本模板针对 CNN backbone (U-Net / Swin-UNETR), 不依赖 SAM2 prompt
#   - 评测 BUSI val.txt (作为 held-out test 集)
# =============================================================================

import os
import sys
sys.path.insert(0, '/home/zhengsongming/jupyterworkspace/UB-MSAM')

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from medpy.metric.binary import hd95
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

from experiments.cross_backbone.train import build_model

# =============================================================================
# 1. 配置 (改这里!)
# =============================================================================
BACKBONE = 'unet'  # 'unet' 或 'swin_unetr'
CKPT_PATH = '/home/zhengsongming/jupyterworkspace/UB-MSAM/runs/EXP_NAME_TO_REPLACE/checkpoints/checkpoint.pt'
DATASET_ROOT = '/home/zhengsongming/jupyterworkspace/datasets/BUSI_for_SAM2'
SIZE = 1024
TEST_SET_FILE = os.path.join(DATASET_ROOT, 'ImageSets', 'val.txt')

# =============================================================================
# 2. 加载模型
# =============================================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading {BACKBONE} from {CKPT_PATH}')
model = build_model(BACKBONE, SIZE).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Model loaded. epoch={ckpt.get("epoch", "?")}')

# =============================================================================
# 3. 指标计算
# =============================================================================
def calculate_metrics(gt_mask, pred_mask):
    gt_b = gt_mask > 0
    pred_b = pred_mask > 0
    inter = np.logical_and(gt_b, pred_b).sum()
    dice = 2 * inter / (gt_b.sum() + pred_b.sum() + 1e-8)
    union = gt_b.sum() + pred_b.sum() - inter
    iou = inter / (union + 1e-8)
    if pred_b.sum() == 0 or gt_b.sum() == 0:
        hd95_score = np.nan
    else:
        hd95_score = hd95(pred_b, gt_b)
    return dice, iou, hd95_score

# =============================================================================
# 4. 推理 + 评测
# =============================================================================
with open(TEST_SET_FILE) as f:
    names = [l.strip() for l in f if l.strip()]
print(f'\n{len(names)} test samples')

img_dir = os.path.join(DATASET_ROOT, 'JPEGImages')
gt_dir = os.path.join(DATASET_ROOT, 'Annotations')
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

results = []
for name in tqdm(names, desc='Evaluating'):
    img_path = os.path.join(img_dir, name, '00000.jpg')
    gt_path = os.path.join(gt_dir, name, '00000.png')
    if not os.path.exists(img_path) or not os.path.exists(gt_path):
        continue

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
    H, W = gt.shape

    # resize + normalize
    img_r = cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
    img_n = (img_r - mean) / std
    img_t = torch.from_numpy(img_n.transpose(2, 0, 1)).float().unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(img_t)
        pred = torch.sigmoid(pred)[0, 0].cpu().numpy()

    # resize 回原图分辨率算指标
    pred = cv2.resize(pred, (W, H), interpolation=cv2.INTER_LINEAR)
    pred_mask = (pred > 0.5).astype(np.uint8) * 255

    dice, iou, h95 = calculate_metrics(gt, pred_mask)
    category = 'benign' if 'benign' in name else 'malignant'
    results.append({'category': category, 'sample_name': name, 'dice': dice, 'iou': iou, 'hd95': h95})

# =============================================================================
# 5. 汇总
# =============================================================================
df = pd.DataFrame(results)
agg = {'dice': ['mean', 'std'], 'iou': ['mean', 'std'], 'hd95': ['mean', 'std'], 'sample_name': ['count']}
cat_summary = df.groupby('category').agg(agg).reset_index()
cat_summary.columns = ['_'.join(c).strip() for c in cat_summary.columns.values]
cat_summary.rename(columns={'category_': 'Category', 'sample_name_count': 'Count'}, inplace=True)

overall = {
    'Category': 'Overall',
    'dice_mean': df['dice'].mean(), 'dice_std': df['dice'].std(),
    'iou_mean': df['iou'].mean(), 'iou_std': df['iou'].std(),
    'hd95_mean': df['hd95'].mean(), 'hd95_std': df['hd95'].std(),
    'Count': len(df),
}
final = pd.concat([cat_summary, pd.DataFrame([overall])], ignore_index=True)

print('\n' + '=' * 80)
print(f'--- {BACKBONE.upper()} | {os.path.basename(os.path.dirname(os.path.dirname(CKPT_PATH)))} ---')
print('=' * 80)
print(final.to_string(index=False))
print('=' * 80)
